## Baseline modeling - LogisticRegression

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix, classification_report, precision_recall_curve, auc

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

In [14]:
PROJECT_ROOT = '../../'
import os

# -----------------------------
# 1. 데이터 불러오기
# -----------------------------
csv_path = os.path.join(PROJECT_ROOT, '00_data', '01_interim', 'train.csv')
data_df = pd.read_csv(csv_path, low_memory=False)

#최종 dataframe
final_df = data_df[['pid']]
final_df.head(2)

,pid
0,10002
1,40002


In [28]:
# -----------------------------
# 2. target / feature 분리
# -----------------------------
X = data_df.drop('is_churned', axis=1)
y = data_df['is_churned']


# -----------------------------
# 3. Train / Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# -----------------------------
# 4. Pipeline (Scaling + Logistic)
# -----------------------------
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])


# -----------------------------
# 5. CV 설정
# -----------------------------
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# -----------------------------
# 6. 평가 지표 설정
# -----------------------------
scoring = {
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision'
}


# -----------------------------
# 7. Cross Validation 수행
# -----------------------------
cv_results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)


# -----------------------------
# 8. CV 결과 출력
# -----------------------------
print("CV Recall:", cv_results['test_recall'].mean())
print("CV F1-score:", cv_results['test_f1'].mean())
print("CV ROC-AUC:", cv_results['test_roc_auc'].mean())
print("CV PR-AUC:", cv_results['test_pr_auc'].mean())


# -----------------------------
# 9. 모델 학습
# -----------------------------
pipeline.fit(X_train, y_train)


# -----------------------------
# 10. 테스트 예측
# -----------------------------
pred = pipeline.predict(X_test)


# -----------------------------
# 11. 테스트 평가
# -----------------------------
print("\nClassification Report")
print(classification_report(y_test, pred))

print("Confusion Matrix")
print(confusion_matrix(y_test, pred))

CV Recall: 0.15454545454545454
CV F1-score: 0.23472661382677704
CV ROC-AUC: 0.6074513075105055
CV PR-AUC: 0.45984322907352676

Classification Report
              precision    recall  f1-score   support

           0       0.67      0.94      0.78       291
           1       0.65      0.20      0.31       165

    accuracy                           0.67       456
   macro avg       0.66      0.57      0.55       456
weighted avg       0.66      0.67      0.61       456

Confusion Matrix
[[273  18]
 [132  33]]
